# 1.RunnableParallel使用技巧

知识点讲解：RunnableParallel 并行运行组件

1. 组件定位：为什么需要 RunnableParallel
   - RunnableParallel 是 LangChain 中封装的支持运行多个 Runnable 的核心类，又称 RunnableMap
   - 官方定位：用于「操作 Runnable 的输出，以匹配序列中下一个 Runnable 的输入」，
     起到并行运行 Runnable 并格式化输出结构的作用
   - 这句定义包含两个并列职责，不要只记住「并行」：
     * 职责一（并行执行）：同时执行多条 Chain，再以字典形式返回各个 Chain 的结果，
       对比每一条链单独执行，效率会高很多 —— 本文件示例演示这一职责
     * 职责二（格式化输出结构）：把上游数据整形成下游 Runnable 需要的输入结构，
       典型写法是 context=检索器、query=透传 —— 见 2.RunnableParallel模拟检索.py
   - 类型签名可理解为 Runnable[Input, dict[str, Any]]

2. 构造方式与参数宽容性
   - 支持传递「字典、函数、映射、键值对数据」等多种方式，底层会执行检测并统一转换为 Runnable
   - 四种等价写法：
     * 关键字参数：RunnableParallel(joke=joke_chain, poem=poem_chain)
     * 字典位置参数：RunnableParallel({"joke": joke_chain, "poem": poem_chain})
     * 混合写法：RunnableParallel({"joke": joke_chain}, poem=poem_chain)
     * 隐式写法：LCEL 管道中直接写裸 dict，由 __or__ / __ror__ 自动转换成 RunnableParallel
   - 因此 value 不必是 Runnable 实例，普通函数、lambda、itemgetter 乃至嵌套 dict 都可以

3. 归一化原理：__init__ 源码（langchain-core/runnables/base.py）
   - 源码要点：
       def __init__(self, steps__=None, **kwargs):
           # 1.检测是否传递字典，如果传递，则提取字段内的所有键值对
           merged = {**steps__} if steps__ is not None else {}
           # 2.传递了键值对，则将键值对更新到 merged 进行合并
           merged.update(kwargs)
           super().__init__(
               # 3.循环遍历 merged 的所有键值对，并将每一个元素转换成 Runnable
               steps__={key: coerce_to_runnable(r) for key, r in merged.items()}
           )
   - 由此得到三条确定性结论：
     * 位置参数 steps__ 与关键字参数 kwargs 会 merge 到同一个 dict，
       混合写法合法；键名冲突时因为 update 在后，kwargs 会覆盖 steps__
     * 每个 value 都经过 coerce_to_runnable 转换，这是「参数宽容性」的底层来源
     * 这也解释了关键字写法与字典写法完全等价：二者最终汇入同一个 merged 字典

4. coerce_to_runnable 的类型分发规则（langchain_core/runnables/base.py）
   - 分发顺序决定了「LCEL 里能塞什么」：
     * 已是 Runnable            -> 原样返回
     * 生成器函数 / 异步生成器   -> 包装成 RunnableGenerator
     * 其他可调用对象           -> 包装成 RunnableLambda
       （lambda、普通函数、itemgetter 都命中这一支）
     * dict                     -> 递归包装成 RunnableParallel
     * 其他类型                 -> 直接抛 TypeError
       （报错信息：Expected a Runnable, callable or dict.）
   - 注意分支顺序：callable 的判断在 dict 之前，
     因此对既可调用又是 dict 子类的对象，会优先当作函数处理
   - 与管道运算符的联动：管道里写裸 dict 之所以能工作，
     正是 __or__ / __ror__ 内部调用了 coerce_to_runnable 把 dict 转成了 RunnableParallel
     （__or__ / __ror__ 的说明见 4-LCEL表达式与Runnable可运行协议）

5. 并行执行原理与性能收益
   - invoke 时内部使用 ThreadPoolExecutor 线程池并发调用各分支
   - ainvoke 时使用 asyncio.gather 并发协程
   - LLM 调用属于 IO 密集型（等待网络响应），并行可显著降低总耗时
   - 总耗时 ≈ max(各分支耗时)，而不是 sum(各分支耗时)

6. 输入分发规则（非常重要）
   - RunnableParallel 把「同一个原始输入」原封不动传给每一个分支
   - 本例中 {"subject": "程序员"} 会同时传给 joke_chain 和 poem_chain
   - 因此各分支的 Prompt 变量名必须能从同一份输入里取到（这里都用 {subject}）

7. 与 RunnableSequence 的对比
   - RunnableSequence（管道 |）：串行，上一步输出是下一步输入，A → B → C
   - RunnableParallel（dict）：并行，同一输入分发到多个分支，{A, B, C} 同时跑
   - 二者可以任意嵌套组合，构成复杂的 DAG 数据流

8. 典型输出示例与观察点
   - 并行两条链后得到一个 dict，key 即分支名：
       {'joke': '为什么程序员总是用尺子测电脑屏幕？因为他们听说了"像素"是屏幕上的一种"尺寸"。',
        'poem': '在代码的海洋里徜徉，
程序员心怀梦想与创意。
键盘敲击是旋律，

                bug 是诗歌的瑕疵。

算法如诗的韵律，
逻辑是句子的构思。

                编程者如诗人般，
创造出数字的奇迹。'}
   - 观察点 1：两个 value 都已是纯字符串，因为每条分支链末尾都挂了 StrOutputParser；
     若去掉 parser，value 会是 AIMessage 对象
   - 观察点 2：输出 dict 的 key 完全由构造时的分支名决定，与执行完成的先后无关，
     即结果顺序是确定的，不受并行调度影响

9. 典型应用场景
   - 一次输入生成多种形式的结果（笑话 + 诗歌 + 摘要）
   - RAG 场景中同时准备 context（检索）与 query（透传）
   - 多路召回：同时查询向量库、关键词库、知识图谱后再融合
   - 多模型对比评测：同一 prompt 同时发给多个 LLM

10. 注意事项
    - 并行分支会同时发起多个 LLM 请求，注意 API 的 QPS/RPM 限流
    - 任意一个分支抛异常，整个 RunnableParallel 都会失败（可配合 with_fallbacks 兜底）
    - 分支越多，token 消耗越大，成本按分支数线性增长
    - 分支内部若共享可变状态，需要注意线程安全问题


In [ ]:
import dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
joke_prompt = ChatPromptTemplate.from_template("请讲一个关于{subject}的冷笑话，尽可能短一些")
poem_prompt = ChatPromptTemplate.from_template("请写一篇关于{subject}的诗，尽可能短一些")
llm = ChatOpenAI(model="deepseek-flash")
parser = StrOutputParser()
joke_chain = joke_prompt | llm | parser
poem_chain = poem_prompt | llm | parser
map_chain = RunnableParallel(joke=joke_chain, poem=poem_chain)

In [ ]:
res = map_chain.invoke({"subject": "程序员"})
print(res)